# FOSS4G 2026 Phase 2 Reproduction Notebook: Multi-Area RAG Evaluation

This notebook reproduces **Phase 2** (4 areas, 130 cases, 4-system comparison) of the FOSS4G 2026 Hiroshima Academic Track paper
*A Systematic Comparison of RAG Architectures for Geographic POI Question Answering Using OpenStreetMap Data*.

- Paper DOI: https://doi.org/10.5194/isprs-archives-L-4-W1-2026-219-2026
- Runtime: Google Colab (**GPU required**. 4-bit quantized inference for `Qwen2.5-7B-Instruct` works on a T4, but an A100 is recommended. Cells that run LLM inference cannot be executed in a local CPU-only environment.)
- If you only want to verify the **GPU-free parts** locally (data loading, test-case loading, evaluator wiring), see `pytest tests/` in this repository.

Note: The RAG systems evaluated here answer Japanese-language questions about POIs in Tokyo, so the system prompts and test questions are intentionally kept in Japanese to preserve the tested behavior. Everything else in this notebook (comments, log output, explanations) is in English.


# Phase 2: Multi-Area RAG Evaluation

**Created**: 2026-02-18  
**Project**: experiments-local-llm  
**Purpose**: Compare RAG systems across 4 areas (Shibuya, Shinjuku, Ikebukuro, Tokyo)

---

## Systems evaluated

| System | Description | Shibuya-only score |
|---------|------|---------------|
| **Hybrid RAG** | Phase 6: Structured RAG | 96.2% |
| **Graph RAG** | Phase 8: Knowledge graph | 76.7% |
| **Adaptive RAG** | Phase 8: Dynamic system routing | 86.1% |
| **Agentic RAG** | Phase 9: LangGraph + ReAct | 87.6% |

## Test cases

- Within-area queries: 80 (4 areas x 20 each)
- Cross-area queries: 20
- Landmark-anchored queries: 15
- Area-detection tests: 15
- **Total: 130**


## 1. Environment Setup


In [ ]:
# Environment check (Colab / local)
import sys
import os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Change this to match where you cloned/extracted this repo on Google Drive
    PROJECT_PATH = '/content/drive/MyDrive/foss4g2026-rag-poi-reproduction'
    sys.path.insert(0, f'{PROJECT_PATH}/src')
    sys.path.insert(0, f'{PROJECT_PATH}/eval')

    # Install required packages
    !pip install -q chromadb sentence-transformers networkx
    !pip install -q transformers accelerate bitsandbytes
    !pip install -q langchain langchain-core langchain-community langchain-chroma langgraph
    !pip install -q matplotlib seaborn pandas numpy tqdm
else:
    PROJECT_PATH = '..'
    sys.path.insert(0, f'{PROJECT_PATH}/src')
    sys.path.insert(0, f'{PROJECT_PATH}/eval')

# Results directory
os.makedirs(f'{PROJECT_PATH}/results', exist_ok=True)

# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: GPU not available. Cells that load the LLM cannot be run from this point on.")

print(f"\nProject path: {PROJECT_PATH}")
print(f"Running in Colab: {IN_COLAB}")


## 2. Load LLM (Qwen2.5-7B-Instruct, 4-bit)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_community.llms import HuggingFacePipeline
import warnings
import gc
warnings.filterwarnings('ignore')

# Clean up VRAM on re-run (in case a previous model is still loaded)
for var_name in ['model', 'text_generation_pipeline', 'llm', 'embeddings']:
    if var_name in dir():
        try:
            del globals()[var_name]
        except KeyError:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM before load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

model_name = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {model_name}...")

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print("Tokenizer loaded")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("Model loaded (4-bit quantized)")

text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.0,
    do_sample=False,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

if torch.cuda.is_available():
    print(f"VRAM after LLM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("LLM pipeline ready")

## 3. Embedding Model (multilingual-e5-base)


In [ ]:
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model_name = "intfloat/multilingual-e5-base"
print(f"Loading embedding model: {embedding_model_name}...")

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print("Embedding model loaded")

## 4. Load POI Data (4 areas + combined)


In [ ]:
import json
from geo_utils import STATIONS, AREA_STATION_MAP, enrich_all_areas

# Area configuration
areas_config = {
    "shibuya": {
        "name": "渋谷駅周辺",
        "station": STATIONS["渋谷駅"]
    },
    "shinjuku": {
        "name": "新宿駅周辺",
        "station": STATIONS["新宿駅"]
    },
    "ikebukuro": {
        "name": "池袋駅周辺",
        "station": STATIONS["池袋駅"]
    },
    "tokyo": {
        "name": "東京駅周辺",
        "station": STATIONS["東京駅"]
    }
}

# Load and concatenate POI data per area
# (To keep the distributed package small, this repo ships poi_{area}.json
#  per area instead of one combined file, and concatenates them here.)
raw_pois = []
for area_key in areas_config:
    area_file = f"{PROJECT_PATH}/data/poi_{area_key}.json"
    print(f"Loading POI data from {area_file}...")
    with open(area_file, "r", encoding="utf-8") as f:
        raw_pois.extend(json.load(f))

# Flatten metadata
flat_pois = []
for poi in raw_pois:
    if "metadata" in poi:
        flat_poi = poi["metadata"].copy()
        flat_pois.append(flat_poi)
    else:
        flat_pois.append(poi)

print(f"Total raw POIs: {len(flat_pois)}")

# Show POI counts by area
area_counts = {}
for poi in flat_pois:
    area_key = poi.get("area_key", "unknown")
    area_counts[area_key] = area_counts.get(area_key, 0) + 1

print("\nPOI counts by area:")
for area, count in sorted(area_counts.items()):
    area_name = areas_config.get(area, {}).get("name", area)
    print(f"  {area_name}: {count}")
print(f"  Total: {sum(area_counts.values())}")


## 5. Build ChromaDB Vector Store (5 collections)


In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
import gc

def create_documents(pois):
    """Create LangChain Documents from a list of POIs."""
    docs = []
    for poi in pois:
        content = f"{poi.get('name', '')} {poi.get('category', '')} {poi.get('description', '')}"
        docs.append(Document(page_content=content, metadata=poi))
    return docs

# Build per-area vector stores plus one combined store
vectorstores = {}
pois_by_area = {}

for area_key in areas_config:
    area_pois = [p for p in flat_pois if p.get("area_key") == area_key]
    pois_by_area[area_key] = area_pois
    docs = create_documents(area_pois)
    vectorstores[area_key] = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        collection_name=f"pois_{area_key}"
    )
    print(f"  {area_key}: {len(docs)} documents")

# Combined collection across all areas
all_docs = create_documents(flat_pois)
vectorstores["all"] = Chroma.from_documents(
    documents=all_docs,
    embedding=embeddings,
    collection_name="pois_all"
)
print(f"  all: {len(all_docs)} documents")

# Free embedding-model VRAM (inference can fall back to CPU)
print("\nVectorstore construction complete")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 6. Add Spatial Info to POIs (enrich_all_areas)


In [ ]:
from geo_utils import enrich_all_areas, enrich_pois_for_area

# Add spatial info to all POIs (computed from each area's reference station)
all_pois_enriched = enrich_all_areas(flat_pois, areas_config)
print(f"Enriched {len(all_pois_enriched)} POIs with spatial info")

# Also keep a per-area breakdown
enriched_by_area = {}
for poi in all_pois_enriched:
    area_key = poi.get("area_key", "unknown")
    enriched_by_area.setdefault(area_key, []).append(poi)

for area_key, area_pois in sorted(enriched_by_area.items()):
    if area_pois:
        avg_dist = sum(p.get("distance_from_station", 0) for p in area_pois) / len(area_pois)
        print(f"  {area_key}: {len(area_pois)} POIs, avg distance: {avg_dist:.0f}m")

## 7. Load Test Cases + Show Statistics


In [ ]:
from test_cases_multi_area import (
    ALL_MULTI_AREA_TEST_CASES,
    get_quick_test_cases,
    get_test_case_stats,
    get_area_tests,
    get_cross_area_tests,
    get_landmark_tests,
    get_detection_tests,
)

print(f"Total test cases: {len(ALL_MULTI_AREA_TEST_CASES)}")

# Show statistics
stats = get_test_case_stats()
print(f"\nBy area:")
for area, count in stats.get('by_area', {}).items():
    print(f"  {area}: {count}")

print(f"\nBy level:")
for level, count in sorted(stats.get('by_level', {}).items()):
    print(f"  L{level}: {count}")

print(f"\nBy query_type:")
for qtype, count in stats.get('by_query_type', {}).items():
    print(f"  {qtype}: {count}")

print(f"\nQuick test cases: {len(get_quick_test_cases())}")

## 8. Initialize RAG Systems (4 systems, with areas_config)


In [ ]:
from structured_rag_system import StructuredRAGSystem
from agentic_rag_system import AgenticRAGSystem
from graph_rag_system import GraphRAGSystem
from adaptive_rag_system import AdaptiveRAGSystem
from agent_tools import set_global_pois_multi_area
from geo_utils import detect_target_area

# Set global POIs (used by Agentic RAG's tools)
set_global_pois_multi_area(all_pois_enriched, areas_config)
print("Global POIs set for multi-area tools")

# Hybrid RAG (StructuredRAG)
print("\nInitializing Hybrid RAG...")
hybrid_system = StructuredRAGSystem(
    model=model,
    tokenizer=tokenizer,
    vectorstore=vectorstores["all"],
    all_pois=all_pois_enriched,
    areas_config=areas_config,
    debug=False
)
print("Hybrid RAG initialized")

# Graph RAG
print("\nInitializing Graph RAG...")
graph_system = GraphRAGSystem(
    areas_config=areas_config,
    all_pois=all_pois_enriched
)
print(f"Graph RAG initialized ({len(graph_system.graphs)} area graphs)")

# Adaptive RAG
print("\nInitializing Adaptive RAG...")
adaptive_system = AdaptiveRAGSystem(
    model=model,
    tokenizer=tokenizer,
    vectorstore=vectorstores["all"],
    all_pois=all_pois_enriched,
    areas_config=areas_config,
    verbose=False
)
print("Adaptive RAG initialized")

# Agentic RAG
print("\nInitializing Agentic RAG...")
agentic_system = AgenticRAGSystem(
    model=model,
    tokenizer=tokenizer,
    model_name=model_name,
    verbose=False,
    max_iterations=5
)
print("Agentic RAG initialized")

# System wrapper functions (conform to the system_fn interface)
# Fall back to detect_target_area() for area detection
def hybrid_fn(question: str) -> dict:
    """Hybrid RAG system_fn wrapper"""
    result = hybrid_system.query(question)
    detected = result.get("detected_area") or detect_target_area(question)
    return {
        "answer": result.get("answer", ""),
        "detected_area": detected,
    }

def graph_fn(question: str) -> dict:
    """Graph RAG system_fn wrapper - GraphRAG only returns context, so generate the LLM answer here.
    NOTE: the system/user prompt below is kept in Japanese on purpose - it is the actual
    prompt sent to the LLM under test, matching the paper's Japanese-language evaluation."""
    graph_result = graph_system.query(question)
    context = graph_result.context

    # Generate the LLM answer (same pattern as adaptive_rag_system._generate_response)
    area_names = "、".join(
        info.get("name", key) for key, info in areas_config.items()
    )
    system_prompt = f"""あなたは東京都内の主要駅周辺エリア（{area_names}）の地理情報に詳しいアシスタントです。
提供された情報に基づいて、正確かつ簡潔に回答してください。
座標情報がある場合は必ず含めてください。
数値データがある場合は具体的な数字を使って回答してください。
情報がない場合は「情報がありません」と正直に回答してください。"""

    prompt = f"""以下の情報を参考にして、質問に回答してください。

{context}

【質問】
{question}

【回答】
上記の情報を基に、具体的な数値や場所名を含めて回答します。"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        parts = response.split("assistant")
        if len(parts) > 1:
            response = parts[-1].strip()

    del inputs, outputs
    torch.cuda.empty_cache()

    detected = detect_target_area(question)
    return {
        "answer": response,
        "detected_area": detected,
    }

def adaptive_fn(question: str) -> dict:
    """Adaptive RAG system_fn wrapper"""
    result = adaptive_system.query(question)
    detected = detect_target_area(question)
    return {
        "answer": result.response,
        "detected_area": detected,
    }

def agentic_fn(question: str) -> dict:
    """Agentic RAG system_fn wrapper"""
    result = agentic_system.query(question)
    detected = result.get("detected_area") or detect_target_area(question)
    return {
        "answer": result.get("answer", ""),
        "detected_area": detected,
    }

# Define all systems (Graph RAG and Adaptive RAG added)
SYSTEMS = {
    "hybrid_rag": hybrid_fn,
    "graph_rag": graph_fn,
    "adaptive_rag": adaptive_fn,
    "agentic_rag": agentic_fn,  # slowest, so run it last
}

print(f"\n{len(SYSTEMS)} systems ready for evaluation")

## 9. Run Quick Test (5 each Shibuya+Shinjuku + 3 cross-area = 13 cases)


In [ ]:
from evaluators_multi_area import MultiAreaEvaluator
from tqdm import tqdm
import gc

def clear_memory():
    """Free VRAM/RAM."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

evaluator = MultiAreaEvaluator(areas_config=areas_config, all_pois=all_pois_enriched)

# Quick-test cases
quick_cases = get_quick_test_cases()
print(f"Quick Test: {len(quick_cases)} cases")
for tc in quick_cases:
    print(f"  {tc.id}: {tc.prompt[:50]}...")

# Run the quick test for each system
quick_results = {}
for sys_name, sys_fn in SYSTEMS.items():
    print(f"\n{'='*60}")
    print(f"Evaluating {sys_name} (Quick Test)")
    print(f"{'='*60}")
    
    results = evaluator.evaluate_all(
        system_name=sys_name,
        system_fn=sys_fn,
        test_cases=quick_cases
    )
    quick_results[sys_name] = results
    
    # Show an immediate summary
    summary = evaluator.generate_summary(results)
    overall = summary["overall"]
    print(f"\n  Success Rate: {overall['success_rate']*100:.1f}%")
    print(f"  Avg Hit Rate: {overall['avg_keyword_hit_rate']:.3f}")
    print(f"  Avg Time: {overall['avg_time_sec']:.1f}s")
    print(f"  Errors: {overall['error_count']}")
    print(f"  Language Issues: {overall['language_issue_count']}")
    print(f"  Avg Composite: {overall['avg_composite_score']:.1f}")
    
    clear_memory()

print("\nQuick Test complete!")

## 10. Run Full Test (with checkpointing)

**Note**: Evaluating all 130 cases x 4 systems can take several hours.  
Set `run_full_test = True` to run it.


In [ ]:
run_full_test = True  # run the full test

full_results = {}

if run_full_test:
    all_cases = ALL_MULTI_AREA_TEST_CASES
    print(f"Full Test: {len(all_cases)} cases x {len(SYSTEMS)} systems")
    
    for sys_name, sys_fn in SYSTEMS.items():
        print(f"\n{'='*60}")
        print(f"Evaluating {sys_name} (Full Test)")
        print(f"{'='*60}")
        
        checkpoint = f"{PROJECT_PATH}/results/checkpoint_{sys_name}.json"
        results = evaluator.evaluate_all(
            system_name=sys_name,
            system_fn=sys_fn,
            test_cases=all_cases,
            checkpoint_file=checkpoint
        )
        full_results[sys_name] = results
        
        summary = evaluator.generate_summary(results)
        overall = summary["overall"]
        print(f"\n  Success Rate: {overall['success_rate']*100:.1f}%")
        print(f"  Avg Hit Rate: {overall['avg_keyword_hit_rate']:.3f}")
        print(f"  Avg Time: {overall['avg_time_sec']:.1f}s")
        
        clear_memory()
    
    print("\nFull Test complete!")
else:
    print("Full Test skipped (set run_full_test = True to run)")
    # Fall back to the quick-test results
    full_results = quick_results

## 10.5 Post-hoc Score Recalculation (multi-dimensional evaluation)

Recompute Phase 9-equivalent multi-dimensional scores from the `answer` text already stored in the restored checkpoint results.
There is no need to re-run the 520 queries.


In [ ]:
# Post-hoc score recalculation (add multi-dimensional scores to existing checkpoint results)
from test_cases_multi_area import ALL_MULTI_AREA_TEST_CASES

for sys_name, results in full_results.items():
    full_results[sys_name] = evaluator.recalculate_scores(results, ALL_MULTI_AREA_TEST_CASES)
    # Show a sample
    sample = full_results[sys_name][0]
    print(f"{sys_name}: composite={sample.composite_score}, reasoning={sample.reasoning_score}, "
          f"evidence={sample.evidence_score}, constraint={sample.constraint_score}")

print("\nMulti-dimensional scores recalculated for all systems")

## 11. Result Analysis: Overall Comparison


In [ ]:
import pandas as pd
import numpy as np

# Overall comparison
comparison = evaluator.compare_systems(full_results)

print("="*90)
print("Overall System Comparison")
print("="*90)

header = (f"{'System':<20} {'Success%':<10} {'AvgHitRate':<11} {'Composite':<10} "
          f"{'CompSucc%':<10} {'AvgTime(s)':<11} {'Errors':<7} {'LangIssue':<10}")
print(header)
print("-"*90)

for sys_name in full_results:
    s = comparison["summaries"][sys_name]["overall"]
    print(f"{sys_name:<20} {s['success_rate']*100:<10.1f} {s['avg_keyword_hit_rate']:<11.3f} "
          f"{s['avg_composite_score']:<10.1f} {s['composite_success_rate']*100:<10.1f} "
          f"{s['avg_time_sec']:<11.1f} {s['error_count']:<7} {s['language_issue_count']:<10}")

# Rankings
print(f"\nRankings (by success rate):")
for i, (name, rate) in enumerate(comparison["rankings"], 1):
    comp = comparison["summaries"][name]["overall"]["avg_composite_score"]
    print(f"  {i}. {name}: success={rate*100:.1f}%, composite={comp:.1f}")

## 12. Result Analysis: By Area / By Subcategory


In [ ]:
# Scores by area
print("="*70)
print("Results by Area")
print("="*70)

for sys_name in full_results:
    summary = comparison["summaries"][sys_name]
    print(f"\n--- {sys_name} ---")
    by_area = summary.get("by_area", {})
    for area, metrics in sorted(by_area.items()):
        print(f"  {area:<15} success={metrics['success_rate']*100:.1f}% hit_rate={metrics['avg_keyword_hit_rate']:.3f} n={metrics['count']}")

# Scores by level
print(f"\n{'='*70}")
print("Results by Level")
print("="*70)

for sys_name in full_results:
    summary = comparison["summaries"][sys_name]
    print(f"\n--- {sys_name} ---")
    by_level = summary.get("by_level", {})
    for level, metrics in sorted(by_level.items()):
        print(f"  L{level}: success={metrics['success_rate']*100:.1f}% hit_rate={metrics['avg_keyword_hit_rate']:.3f} n={metrics['count']}")

# Scores by subcategory
print(f"\n{'='*70}")
print("Results by Subcategory")
print("="*70)

for sys_name in full_results:
    summary = comparison["summaries"][sys_name]
    print(f"\n--- {sys_name} ---")
    by_sub = summary.get("by_subcategory", {})
    for sub, metrics in sorted(by_sub.items()):
        print(f"  {sub:<25} success={metrics['success_rate']*100:.1f}% n={metrics['count']}")

## 13. Result Analysis: Cross-Area + Area-Detection Accuracy


In [ ]:
# Cross-area results
print("="*70)
print("Cross-Area Query Results")
print("="*70)

for sys_name in full_results:
    summary = comparison["summaries"][sys_name]
    cross = summary.get("cross_area", {})
    print(f"  {sys_name}: success={cross.get('success_rate', 0)*100:.1f}% n={cross.get('count', 0)} avg_time={cross.get('avg_time_sec', 0):.1f}s")

# Area-detection accuracy
print(f"\n{'='*70}")
print("Area Detection Accuracy")
print("="*70)

for sys_name in full_results:
    summary = comparison["summaries"][sys_name]
    detection = summary.get("area_detection", {})
    print(f"  {sys_name}: accuracy={detection.get('accuracy', 0)*100:.1f}% ({detection.get('correct', 0)}/{detection.get('total', 0)})")

# Area consistency (score variance)
print(f"\n{'='*70}")
print("Area Consistency (lower variance = more consistent)")
print("="*70)

for sys_name, variance in comparison.get("area_consistency", {}).items():
    print(f"  {sys_name}: variance={variance:.4f}")

## 14. Visualization


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

# --- Fig 1: System Comparison Bar Chart ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Overall success rate
sys_names = list(full_results.keys())
success_rates = [comparison["summaries"][s]["overall"]["success_rate"] * 100 for s in sys_names]
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12'][:len(sys_names)]

axes[0].barh(sys_names, success_rates, color=colors, alpha=0.8)
axes[0].set_xlabel('Success Rate (%)')
axes[0].set_title('Overall Success Rate by System')
axes[0].set_xlim([0, 110])
for i, v in enumerate(success_rates):
    axes[0].text(v + 2, i, f'{v:.1f}%', va='center')

# Average execution time
avg_times = [comparison["summaries"][s]["overall"]["avg_time_sec"] for s in sys_names]
axes[1].barh(sys_names, avg_times, color=colors, alpha=0.8)
axes[1].set_xlabel('Avg Time (seconds)')
axes[1].set_title('Average Execution Time by System')
for i, v in enumerate(avg_times):
    axes[1].text(v + 0.5, i, f'{v:.1f}s', va='center')

plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/results/phase9b_overall_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Fig 2: Area-wise Success Rate ---
fig, ax = plt.subplots(figsize=(14, 8))

area_keys = sorted(areas_config.keys())
x = np.arange(len(area_keys))
width = 0.8 / len(sys_names)

for i, sys_name in enumerate(sys_names):
    by_area = comparison["summaries"][sys_name].get("by_area", {})
    rates = [by_area.get(area, {}).get('success_rate', 0) * 100 for area in area_keys]
    offset = (i - len(sys_names)/2 + 0.5) * width
    bars = ax.bar(x + offset, rates, width, label=sys_name, color=colors[i], alpha=0.8)

ax.set_xlabel('Area')
ax.set_ylabel('Success Rate (%)')
ax.set_title('Success Rate by Area and System')
ax.set_xticks(x)
ax.set_xticklabels([areas_config[a]["name"] for a in area_keys])
ax.legend()
ax.set_ylim([0, 110])

plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/results/phase9b_area_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Fig 3: Level-wise Success Rate ---
fig, ax = plt.subplots(figsize=(12, 6))

levels = [1, 2, 3, 4, 5]
x = np.arange(len(levels))

for i, sys_name in enumerate(sys_names):
    by_level = comparison["summaries"][sys_name].get("by_level", {})
    rates = [by_level.get(l, {}).get('success_rate', 0) * 100 for l in levels]
    offset = (i - len(sys_names)/2 + 0.5) * width
    ax.bar(x + offset, rates, width, label=sys_name, color=colors[i], alpha=0.8)

level_names = ['L1 Basic', 'L2 Spatial', 'L3 Constraint', 'L4 Decision', 'L5 Advanced']
ax.set_xlabel('Level')
ax.set_ylabel('Success Rate (%)')
ax.set_title('Success Rate by Level and System')
ax.set_xticks(x)
ax.set_xticklabels(level_names)
ax.legend()
ax.set_ylim([0, 110])

plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/results/phase9b_level_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualizations saved")

## 15. Save Results (JSON + text summary)


In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Convert numpy types to native Python types
def convert_to_serializable(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(i) for i in obj]
    return obj

# Save JSON results
results_data = {
    "timestamp": timestamp,
    "test_count": len(next(iter(full_results.values()))),
    "is_quick_test": not run_full_test if 'run_full_test' in dir() else True,
    "systems": {}
}

for sys_name, results in full_results.items():
    summary = evaluator.generate_summary(results)
    results_data["systems"][sys_name] = {
        "summary": convert_to_serializable(summary),
        "results": [convert_to_serializable(r.to_dict()) for r in results]
    }

results_data["comparison"] = convert_to_serializable(comparison)

output_file = f"{PROJECT_PATH}/results/phase9b_evaluation_{timestamp}.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results_data, f, ensure_ascii=False, indent=2)
print(f"Results saved to {output_file}")

# Save text summary
summary_file = f"{PROJECT_PATH}/results/phase9b_summary_{timestamp}.txt"
with open(summary_file, 'w', encoding='utf-8') as f:
    f.write("Phase 9-B: Multi-Area RAG Evaluation Summary\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Test Cases: {results_data['test_count']}\n")
    f.write(f"Quick Test: {results_data['is_quick_test']}\n\n")
    
    f.write("Overall Results:\n")
    f.write("-" * 60 + "\n")
    for sys_name in full_results:
        s = comparison["summaries"][sys_name]["overall"]
        f.write(f"{sys_name}: success={s['success_rate']*100:.1f}% ")
        f.write(f"hit_rate={s['avg_keyword_hit_rate']:.3f} ")
        f.write(f"time={s['avg_time_sec']:.1f}s\n")
    
    f.write("\nArea-wise Results:\n")
    f.write("-" * 60 + "\n")
    for sys_name in full_results:
        f.write(f"\n{sys_name}:\n")
        by_area = comparison["summaries"][sys_name].get("by_area", {})
        for area, metrics in sorted(by_area.items()):
            f.write(f"  {area}: {metrics['success_rate']*100:.1f}% (n={metrics['count']})\n")
    
    f.write("\nCross-Area Results:\n")
    f.write("-" * 60 + "\n")
    for sys_name in full_results:
        cross = comparison["summaries"][sys_name].get("cross_area", {})
        f.write(f"  {sys_name}: {cross.get('success_rate', 0)*100:.1f}% (n={cross.get('count', 0)})\n")
    
    f.write("\nArea Detection Accuracy:\n")
    f.write("-" * 60 + "\n")
    for sys_name in full_results:
        det = comparison["summaries"][sys_name].get("area_detection", {})
        f.write(f"  {sys_name}: {det.get('accuracy', 0)*100:.1f}% ({det.get('correct', 0)}/{det.get('total', 0)})\n")

print(f"Summary saved to {summary_file}")
print("\nEvaluation complete!")

## 16. Conclusions and Next Steps

### Summary of evaluation results

Phase 2 evaluated RAG systems for broad-area coverage across 4 areas (Shibuya, Shinjuku, Ikebukuro, Tokyo) using 130 test cases.

### Key findings

1. **Area generalization**: how the structured pipeline performs outside Shibuya
2. **Cross-area**: how well each system handles queries that compare across areas
3. **Area detection**: accuracy of identifying the target area from the question text
4. **Area consistency**: whether the same question pattern is handled uniformly across areas

### Recommendations for Phase 10 (original research roadmap)

These were the original next steps identified in the source research project (nationwide expansion, Phase 10). They are kept here for historical context; this reproduction package covers Phase 2 only.

- Select a base RAG approach
- Assess whether area-detection logic needs improvement
- Prioritize cross-area support
- Design the migration to PostGIS/Supabase
